# AI Dermatology Assistant

### Multi-Class Skin Disease Prediction using Machine Learning and LLM

**Industry:** Healthcare (Dermatology)

## Problem Statement

Skin diseases often have similar symptoms, making early identification difficult for people without medical knowledge. This project develops a multi-class machine learning model that predicts the most likely skin disease based on symptoms entered by the user. The predicted disease will later be explained using an LLM and displayed through a Streamlit web application.

## Objective

* Predict common skin diseases using patient symptoms.
* Compare multiple machine learning algorithms.
* Save the best-performing model.
* Build a Streamlit interface for users.
* Integrate an LLM to explain the prediction and provide basic skincare guidance.


In [1]:
# ============================
# Import Required Libraries
# ============================

import pandas as pd
import numpy as np

# Visualization Libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

# Machine Learning Libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Save Model
import joblib

In [ ]:
# ======================================
# Load Dermatology Dataset
# ======================================

# Read the CSV file from the data folder
df = pd.read_csv("../data/dermatology_capstone_dataset.csv")

In [3]:
# Display first 5 rows
df.head(5)

,patient_id,gender,body_area,itching_level,rash_type,skin_peeling,burning_sensation,swelling,fever,skin_color_change,...,stress_level,skin_type,disease,age,bmi,body_temperature,symptom_duration_days,severity_score,sleep_hours,water_intake_liters
0,6d05315f-022d-485b-a809-ab4e7ec3f2a3,Female,Face,Mild,Red Rash,Yes,No,Yes,No,No,...,High,Dry,Rosacea,40.0,18.0,98.7,30.0,5.0,7.3,3.1
1,35ced9a2-0e8d-414a-b3c2-1e58f8199e69,Male,Arms,Mild,Red Rash,Yes,No,Yes,No,Yes,...,High,Oily,Urticaria,47.0,24.7,97.6,60.0,7.0,5.5,1.6
2,ad389eae-160c-43a1-b21d-7d62fdf7ee1f,Other,Face,Severe,Red Rash,Yes,No,Yes,No,Yes,...,High,Sensitive,Rosacea,51.0,29.1,98.5,90.0,5.0,NaN,2.5
3,c261d043-e7c5-4236-be9e-0213c6cbb035,Male,Feet,Severe,Circular Rash,No,No,Yes,No,Yes,...,High,Normal,Fungal Infection,38.0,26.2,98.5,5.0,5.0,7.1,2.0
4,70fad4ca-165b-4dc5-a9d9-e2c097edd257,Male,Feet,Mild,White Patch,Yes,Yes,Yes,No,No,...,High,Oily,Acne,26.0,21.9,99.4,2.0,8.0,5.9,1.3


In [4]:
# ======================================
# Shape of Dataset
# ======================================

df.shape

(8093, 27)

In [5]:
# ======================================
# Size of Dataset
# ======================================

df.size

218511

In [6]:
# ======================================
# Dataset Information
# ======================================

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8093 entries, 0 to 8092
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   patient_id             8093 non-null   str    
 1   gender                 8071 non-null   str    
 2   body_area              8052 non-null   str    
 3   itching_level          7489 non-null   str    
 4   rash_type              7863 non-null   str    
 5   skin_peeling           7998 non-null   str    
 6   burning_sensation      7925 non-null   str    
 7   swelling               8093 non-null   str    
 8   fever                  8093 non-null   str    
 9   skin_color_change      8093 non-null   str    
 10  pain                   8093 non-null   str    
 11  dry_skin               8093 non-null   str    
 12  pus_or_discharge       8093 non-null   str    
 13  family_history         8034 non-null   str    
 14  allergy_history        7806 non-null   str    
 15  recent_new_prod

In [7]:
# ======================================
# Statistical Summary
# ======================================

df.describe()

,age,bmi,body_temperature,symptom_duration_days,severity_score,sleep_hours,water_intake_liters
count,8074.000000,8056.000000,8019.000000,7912.000000,8030.000000,7964.000000,8001.000000
mean,32.508174,23.639424,98.643023,32.063195,6.202366,6.798707,2.298438
std,12.796891,4.069553,0.619984,28.825381,1.913504,1.291647,0.694724
min,1.000000,15.500000,96.700000,2.000000,1.000000,3.500000,0.500000
25%,24.000000,20.800000,98.300000,5.000000,5.000000,5.900000,1.800000
50%,33.000000,23.600000,98.600000,18.000000,6.000000,6.800000,2.300000
75%,41.000000,26.300000,99.000000,45.000000,8.000000,7.700000,2.800000
max,83.000000,39.500000,102.300000,90.000000,10.000000,10.000000,4.900000


In [8]:
# ======================================
# Missing Values
# ======================================

df.isnull().sum()

patient_id                 0
gender                    22
body_area                 41
itching_level            604
rash_type                230
skin_peeling              95
burning_sensation        168
swelling                   0
fever                      0
skin_color_change          0
pain                       0
dry_skin                   0
pus_or_discharge           0
family_history            59
allergy_history          287
recent_new_product        52
humidity_exposure        104
stress_level             146
skin_type                 89
disease                    0
age                       19
bmi                       37
body_temperature          74
symptom_duration_days    181
severity_score            63
sleep_hours              129
water_intake_liters       92
dtype: int64

In [9]:
# ======================================
# Missing Value Percentage
# ======================================

round((df.isnull().sum()/len(df))*100,2)

patient_id               0.00
gender                   0.27
body_area                0.51
itching_level            7.46
rash_type                2.84
skin_peeling             1.17
burning_sensation        2.08
swelling                 0.00
fever                    0.00
skin_color_change        0.00
pain                     0.00
dry_skin                 0.00
pus_or_discharge         0.00
family_history           0.73
allergy_history          3.55
recent_new_product       0.64
humidity_exposure        1.29
stress_level             1.80
skin_type                1.10
disease                  0.00
age                      0.23
bmi                      0.46
body_temperature         0.91
symptom_duration_days    2.24
severity_score           0.78
sleep_hours              1.59
water_intake_liters      1.14
dtype: float64

In [10]:
# ======================================
# Duplicate Records
# ======================================

df.duplicated().sum()

np.int64(85)

In [11]:
# ======================================
# Unique Diseases
# ======================================

df["disease"].value_counts()

disease
Acne                  2094
Eczema                1478
Fungal Infection      1172
Psoriasis              905
Contact Dermatitis     718
Ringworm               613
Rosacea                425
Urticaria              354
Vitiligo               223
Chickenpox             111
Name: count, dtype: int64

## Data Preparation (Transformation)

In [17]:
# ======================================
# Create a Working Copy of Dataset
# ======================================

df_clean = df.copy()

In [18]:
df_clean.shape

(8093, 27)

##### A copy of the original dataset is created to preserve the raw data. All preprocessing and cleaning steps will be performed on df_clean.

In [19]:
# ======================================
# Remove Duplicate Records
# ======================================

df_clean.drop_duplicates(inplace = True)

In [20]:
df_clean.duplicated().sum()

np.int64(0)

##### Duplicate patient records can bias the machine learning model. Therefore, duplicate entries are removed before preprocessing.

In [21]:
# ======================================
# Missing Values Before Treatment
# ======================================

missing_df = pd.DataFrame({
    "Missing Values": df_clean.isnull().sum(),
    "Percentage": round((df_clean.isnull().sum()/len(df_clean))*100,2)
})

missing_df = missing_df[missing_df["Missing Values"]>0]

missing_df.sort_values(by="Missing Values", ascending=False)

,Missing Values,Percentage
itching_level,600,7.49
allergy_history,287,3.58
rash_type,230,2.87
symptom_duration_days,181,2.26
burning_sensation,168,2.10
stress_level,146,1.82
sleep_hours,129,1.61
humidity_exposure,104,1.30
skin_peeling,95,1.19
water_intake_liters,92,1.15


##### This table shows the number and percentage of missing values remaining after removing duplicate records

In [22]:
df_clean["gender"].value_counts()

gender
Female    2731
Male      2621
Other     2607
female      15
MALE        12
Name: count, dtype: int64

In [23]:
# ======================================
# Remove rows from columns having
# missing value percentage less than 2.5%
# ======================================

# Calculate missing percentage
missing_percent = (df_clean.isnull().sum() / len(df_clean)) * 100

# Columns with less than 2.5% missing values
low_null_cols = missing_percent[missing_percent < 2.5].index

In [24]:
# Remove rows containing null values in those columns
df_clean = df_clean.dropna(subset=low_null_cols)

In [25]:
df_clean.shape

(6735, 27)

##### First, the percentage of missing values is calculated for each column. Columns having less than 2.5% missing values are identified. Since the amount of missing data is very small, rows containing null values in these columns are removed using dropna(subset=...). This minimizes information loss while avoiding unnecessary imputation.

In [26]:
df_clean.isnull().sum()

patient_id                 0
gender                     0
body_area                  0
itching_level            500
rash_type                196
skin_peeling               0
burning_sensation          0
swelling                   0
fever                      0
skin_color_change          0
pain                       0
dry_skin                   0
pus_or_discharge           0
family_history             0
allergy_history          235
recent_new_product         0
humidity_exposure          0
stress_level               0
skin_type                  0
disease                    0
age                        0
bmi                        0
body_temperature           0
symptom_duration_days      0
severity_score             0
sleep_hours                0
water_intake_liters        0
dtype: int64

In [27]:
import numpy as np

cat_cols = df_clean.select_dtypes(include=["object", "string"]).columns

for col in cat_cols:
    # Remove spaces
    df_clean[col] = df_clean[col].str.strip()

    # Convert string "nan" into actual missing value
    df_clean[col] = df_clean[col].replace("nan", np.nan)

    # Fill ONLY missing values
    df_clean[col] = df_clean[col].fillna(df[col].mode()[0])

##### Missing values in categorical columns are replaced with the mode (most frequently occurring value) because it preserves the existing category distribution without introducing new categories.

In [29]:
df_clean.isnull().sum()


patient_id               0
gender                   0
body_area                0
itching_level            0
rash_type                0
skin_peeling             0
burning_sensation        0
swelling                 0
fever                    0
skin_color_change        0
pain                     0
dry_skin                 0
pus_or_discharge         0
family_history           0
allergy_history          0
recent_new_product       0
humidity_exposure        0
stress_level             0
skin_type                0
disease                  0
age                      0
bmi                      0
body_temperature         0
symptom_duration_days    0
severity_score           0
sleep_hours              0
water_intake_liters      0
dtype: int64

In [30]:
# ======================================
# Clean Text Columns
# ======================================

text_columns = df_clean.select_dtypes(include="object").columns

for col in text_columns:
    df_clean[col] = df_clean[col].astype(str).str.strip()

In [31]:
# Fix inconsistent gender values
df_clean["gender"].value_counts()

gender
Female    2314
Other     2215
Male      2181
female      15
MALE        10
Name: count, dtype: int64

In [32]:
df_clean["gender"] = df_clean["gender"].replace({
    "female": "Female",
    "MALE": "Male"
})

df_clean["gender"].value_counts()

gender
Female    2329
Other     2215
Male      2191
Name: count, dtype: int64

##### Text values are standardized by removing leading/trailing spaces and correcting inconsistent spellings. This prevents the model from treating "Female" and "female" as different categories.

In [34]:
# ======================================
# Save Clean Dataset
# ======================================

df_clean.to_csv("../data/dermatology_cleaned_dataset.csv", index=False)

print("Clean dataset saved successfully!")

Clean dataset saved successfully!
